In [1]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
from scipy import ndimage
import numpy as np
import glob
import os
import json

from PIL import Image, ImageFont, ImageDraw

existing QApplication: 0


global modules:create qapp
 /volatile/ad279118/brainvisa/.pixi/envs/default/share/anatomist-6.0/python_plugins
home   modules: /home/ad279118/.anatomist/python_plugins
done
Starting Anatomist.....
config file : /home/ad279118/.anatomist/config/settings.cfg
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module measure
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


In [10]:
subject = "197550"
side = "L"

middle_view = [0.532180190086365, -0.335046976804733, -0.448673963546753, 0.634995579719543]
side_view = [0.532180190086365, 0.335046976804733, 0.448673963546753, 0.634995579719543]

sources = glob.glob(f'/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/*')
file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'

In [11]:
def to_bucket(obj):
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck


def crop_mask(file_src, file_cropped, mask):
    """Crops according to mask"""
    volume = aims.read(file_src)
    print(np.count_nonzero(volume.np))
    if mask:
        mask = aims.read(mask)
        arr = volume.np
        arr_mask = np.asarray(mask)
        arr[arr_mask == 0] = 0
        print(np.count_nonzero(volume.np))
    aims.write(volume, file_cropped)

def build_gradient(pal):
    """Build a gradient palette for Anatomist visualization."""
    gw = ana.cpp.GradientWidget(None, 'gradientwidget', pal.header()['palette_gradients'])
    gw.setHasAlpha(True)
    nc = pal.shape[0]
    rgbp = gw.fillGradient(nc, True)
    rgb = rgbp.data()
    npal = pal.np['v']
    pb = np.frombuffer(rgb, dtype=np.uint8).reshape((nc, 4))
    npal[:, 0, 0, 0, :] = pb
    # Convert BGRA to RGBA
    npal[:, 0, 0, 0, :3] = npal[:, 0, 0, 0, :3][:, ::-1]
    pal.update()

def create_grid(image_files, n_cols, out_path, title=None, subject_names=None):
    # load all images
    imgs = [Image.open(f) for f in image_files]
    # calculate max width and height
    w = max(im.width for im in imgs)
    h = max(im.height for im in imgs)
    # calculate number of rows
    n_rows = (len(imgs) + n_cols - 1) // n_cols

    title_h = 0
    legend_h = 0
    font_size = 36
    if title or subject_names:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
        if title:
            title_h = font.getbbox(title)[3] - font.getbbox(title)[1] + 10  # add margin
        if subject_names:
            legend_h = font.getbbox("Test")[3] - font.getbbox("Test")[1] + 15  # idem

    # create a new blank image with space for title and legend
    grid = Image.new('RGB', (n_cols * w, n_rows * (h + legend_h) + title_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)

    # Draw title
    if title:
        bbox = draw.textbbox((0, 0), title, font=font)  
        text_w = bbox[2] - bbox[0]
        x = (grid.width - text_w) // 2
        draw.text((x, 5), title, fill=(0, 0, 0), font=font)

    # Paste images and draw subject names
    for idx, im in enumerate(imgs):
        i, j = divmod(idx, n_cols)
        x0 = j * w
        y0 = title_h + i * (h + legend_h)
        grid.paste(im, (x0, y0))

        if subject_names:
            subj = subject_names[idx]
            text_bbox = draw.textbbox((0, 0), subj, font=font)
            text_w = text_bbox[2] - text_bbox[0]
            draw.text((x0 + (w - text_w) // 2, y0 + h-50), subj, fill=(0, 0, 0), font=font)

    grid.save(out_path)
    print(f"Snapshot of the block available at {out_path}")

In [12]:
PATH_LIST_REGIONS = "/neurospin/dico/data/deep_folding/current/sulci_regions_champollion_V1.json"
with open(PATH_LIST_REGIONS) as f:
    d = json.load(f)

list_regions = [w.replace('_left', '').replace('_right', '') for w in list(d['brain'].keys())]
list_regions = list(set(list_regions))
len(list_regions)
list_regions
list_region999 = []
list_regions =['S.F.marginal-S.F.inf.ant.',
    'S.F.int.-S.R.',
    'S.Or.',
    'S.Or.-S.Olf.',
    'S.F.inter.-S.F.sup.',
    'S.F.median-S.F.pol.tr.-S.F.sup.',
    'S.F.int.-F.C.M.ant.',
    'S.F.inf.-BROCA-S.Pe.C.inf.',
    'S.Pe.C.',
    'F.C.L.p.-subsc.-F.C.L.a.-INSULA.',
    'S.C.-S.Pe.C.',
    'S.Call.',
    'S.C.-sylv.',
    'S.C.-S.Po.C.',
    'S.T.s.',
    'F.C.M.post.-S.p.C.',
    'S.Call.-S.s.P.-S.intraCing.',
    'S.T.i.-S.T.s.-S.T.pol.',
    'S.Po.C.',
    'F.Coll.-S.Rh.',
    'S.T.i.-S.O.T.lat.',
    'S.s.P.-S.Pa.int.',
    'F.I.P.-F.I.P.Po.C.inf.',
    'S.T.s.br.',
    'Lobule_parietal_sup.',
    'Sc.Cal.-S.Li.',
    'F.P.O.-S.Cu.-Sc.Cal.',
    'OCCIPITAL']

In [13]:
"""
bv bash
cd /volatile/ad279118/deep_folding
. venv/bin/activate
pip install -e .
cd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils
python3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report/197550' -t '/volatile/ad279118/Figures_report/197550'
"""

"\nbv bash\ncd /volatile/ad279118/deep_folding\n. venv/bin/activate\npip install -e .\ncd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils\npython3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report/197550' -t '/volatile/ad279118/Figures_report/197550'\n"

In [14]:
w = a.createWindow("3D")
w2 = a.createWindow("3D")
dic_windows = {}

### To load the white mesh of the subject

In [15]:
# to plot the whole sulcal skeleton of the subject
dic_windows[f'source_{subject}'] = a.loadObject(file_src)
dic_windows[f'source_{subject}'].loadReferentialFromHeader()
dic_windows[f'fusion_{subject}'] = a.fusionObjects(objects=[dic_windows[f'source_{subject}']], method='VolumeRenderingFusionMethod')
w.addObjects(dic_windows[f'fusion_{subject}'])

# to plot the white mesh of the same subject
path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject}/t1mri/BL'
dic_windows[f'white_{subject}'] = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_{side}white.gii')
dic_windows[f'white_{subject}'].loadReferentialFromHeader()

nifti transfo: 1


ATransformSet::unregisterObserver: ref 0x5d1c52be5280 not found


nifti transfo: 2


In [16]:
def print_camera_infos(window):
    try:
        info = window.getInfos()
        quat = info.get('view_quaternion', None)
        zoom = info.get('zoom', None)
        print(f"  Quaternion : {quat}")
        print(f"  Zoom       : {zoom}")
    except Exception as e:
        print(f"Erreur pour la fenêtre : {e}")
        
print_camera_infos(w2)

  Quaternion : [0.70710676908493, 0, 0, 0.70710676908493]
  Zoom       : 1


### To load the buckets of each masked region

In [19]:
counter = 0
for source in sources:
    region = source.replace('/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/', '')
    if region in list_regions:
        list_region999.append(region)
        mask_path = f'{source}/mask/{side}mask_skeleton.nii.gz'
        file_cropped = f'/volatile/ad279118/Figures_report/{subject}/{subject}_{region}_{side}_cropped_skeleton.nii.gz'
        crop_mask(file_src, file_cropped, mask_path)
        #dic_windows[f'vol_{region}'] = aims.read(file_cropped)
        #dic_windows[f'a_obj_{region}'] = a.toAObject(dic_windows[f'vol_{region}'])
        #dic_windows[f'fusion_{region}'] = a.fusionObjects(objects=[dic_windows[f'a_obj_{region}']], method='VolumeRenderingFusionMethod')
        #w.addObjects(dic_windows[f'fusion_{region}'])

        dic_windows[f'obj_init_{region}'] = a.loadObject(file_cropped)
        dic_windows[f'bcks_{region}'] = to_bucket(dic_windows[f'obj_init_{region}'])
        w2.addObjects(dic_windows[f'bcks_{region}'])

        #dic_windows[f'obj_init_{region}'] = a.loadObject(file_src)
        #dic_windows[f'bcks_{region}'] = to_bucket(dic_windows[f'obj_init_{subject}'])
        #w2.addObjects(dic_windows[f'bcks_{region}'])

        """
        path_to_bck = f"/volatile/ad279118/Figures_report/197550/{subject}_{region}_{side}_cropped_skeleton.bck"
        dic_windows[f'bcks_{region}'] = a.loadObject(path_to_bck)
        dic_windows[f'bcks_{region}'].loadReferentialFromHeader()
        w2.addObjects(dic_windows[f'bcks_{region}'])"""

        save_dir = "/volatile/ad279118/2026_Noillopmahc"
        w2.setHasCursor(0)
        w2.camera(view_quaternion=side_view if side == "L" else middle_view, zoom=0.67)
        recon_fname = f"{subject}_{side}_skeleton_{counter}.png"
        recon_img_path = os.path.join(save_dir, recon_fname)
        #w2.snapshot(recon_img_path, width=1200, height=900)
        counter+=1

10691
1137
10691
1450
10691
1394
10691
1436
10691
1954
10691
1554
10691
348
10691
1196
10691
637
10691
1267
10691
642
10691
1757
10691
725
10691
820
10691
2421
10691
913
10691
2135
10691
2007
10691
1379
10691
1681
10691
2280
10691
1287
10691
1560
10691
1904
10691
1169
10691
1847
10691
1254
10691
479


Position : 151.015, 97.0945, 63.9764, 0
no position could be read at 272, 232
Position : 162.032, 104.58, 107.645, 0
Position : 161.016, 118.582, 79.582, 0
Position : 132.902, 117.271, 128.765, 0
Position : 146.902, 42.8446, 106.799, 0
Position : 118.901, 65.9543, 68.4981, 0
Position : 141.101, 48.8479, 102.519, 0
Position : 144.93, 173.567, 107.784, 0
Position : 152.929, 112.408, 145.353, 0
no position could be read at 126, 222
no position could be read at 279, 187
no position could be read at 114, 186
Position : 117.004, 171.333, 56.3145, 0
Position : 151.026, 114.578, 97.2025, 0
Position : 143.04, 80.392, 116.551, 0
Position : 133.045, 125.098, 123.338, 0
Position : 145.016, 175.773, 123.587, 0
Position : 163.016, 119.448, 106.077, 0
Position : 139.016, 113.953, 102.449, 0
Position : 161.016, 95.9411, 100.784, 0
no position could be read at 372, 528
Position : 135.016, 47.8291, 80.962, 0
no position could be read at 280, 410
Position : 147.016, 44.7196, 103.676, 0
no position could 

snap 1 : OpenGL error: invalid operation


no position could be read at 221, 166
no position could be read at 319, 178
Position : 143.144, 156.895, 89.0611, 0
Position : 157.144, 162.854, 96.1913, 0
Position : 151.103, 162.93, 94.8023, 0
Position : 122.975, 175.968, 76.496, 0
Position : 151.041, 170.527, 115.13, 0
Position : 99.7556, 180.619, 86.4724, 0
Position : 123.131, 184.838, 97.4869, 0
Position : 125.049, 189.498, 121.107, 0
Position : 128.977, 192.491, 122.842, 0
no position could be read at 198, 509
no position could be read at 447, 545
Position : 165.428, 135.771, 73.2355, 0
no position could be read at 729, 539
no position could be read at 659, 462
no position could be read at 474, 423
Position : 118.899, 191.813, 119.517, 0
Position : 162.878, 118.066, 73.3471, 0
Position : 154.972, 86.3534, 80.1228, 0
Position : 160.328, 96.6752, 100.576, 0
no position could be read at 465, 606
Position : 121.349, 148.589, 101.406, 0
no position could be read at 1241, 924


QLayout: Attempting to add QLayout "" to QWidget "", which already has a layout


Position : 122.987, 89.0778, 54.1374, 0
Position : 122.987, 89.0778, 54.1374, 0
Position : 122.98, 97.9036, 53.7301, 0
Position : 154.98, 109.152, 64.0036, 0
Position : 142.987, 113.501, 58.2124, 0
Position : 124.987, 88.7583, 57.1261, 0
Position : 124.98, 97.0807, 51.4981, 0
Position : 128.987, 115.525, 47.6679, 0
Position : 139.012, 88.986, 72.2168, 0
no position could be read at 1110, 564
Position : 125.013, 80.9705, 51.9024, 0
Position : 123.013, 120.945, 51.9605, 0
Position : 116.987, 171.104, 55.4118, 0
no position could be read at 1512, 200
Position : 124.987, 131.721, 49.7715, 0
Position : 124.987, 88.2904, 57.8762, 0
Position : 131.013, 140.908, 52.6704, 0
no position could be read at 317, 126
Position : 124.987, 119.088, 46.9901, 0
Position : 124.987, 119.088, 46.9901, 0
Position : 124.987, 119.493, 47.5974, 0
Position : 125.001, 131.701, 52.9998, 0
Position : 132.987, 152.781, 65.6885, 0
Position : 140.987, 105.463, 102.183, 0
Position : 164.987, 101.468, 111.714, 0
Position

: 

### To load the reconstructed hemisphere for a list of subjects
Needs the code decode_global_brain.py to have run before

In [84]:
nb_columns = 4
block = a.createWindowsBlock(nb_columns)
dic_windows2 = {}
pal = a.createPalette('VR-palette')
pal.header()['palette_gradients'] = "1;1#0;1;1;0#0.994872;0#0;0;0.635897;0.266667;1;1"
# 0;1;0.182051;1;0.248718;0;0.628205;0;0.75641;1;1;1#0;1;0.171795;1;0.235897;0;0.487179;0;0.646154;0.977778;1;1#0.220513;1;0.24359;0.822222;0.487179;0.822222;1;1#0;0;0.148718;1;0.828205;1;1;0
build_gradient(pal)
subjects = [
    103414,
    103515,
    103818,
    107220
]

for subject in subjects:
    #file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'
    file_recon = f"/volatile/ad279118/Figures_report/global_reconstruction/{side}_{subject}_decoded.nii.gz"
    #print(os.path.exists(file_recon))
    # load the input 
    dic_windows2[f'w_init_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_init_{subject}'] = a.loadObject(file_src)
    dic_windows2[f'buck_init_{subject}'] = to_bucket(dic_windows2[f'obj_init_{subject}'])
    dic_windows2[f'fusion__init_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_init_{subject}']], 
                                                            method='VolumeRenderingFusionMethod')
    dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion__init_{subject}'])

    # load the reconstruction
    dic_windows2[f'w_recon_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_recon_{subject}'] = a.loadObject(file_recon)
    dic_windows2[f'fusion_recon_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_recon_{subject}']], 
                                                                        method='VolumeRenderingFusionMethod')
    dic_windows2[f'fusion_recon_{subject}'].setPalette('VR-palette', 
                                                    minVal=0, 
                                                    maxVal=0.5, 
                                                    absoluteMode=True)
    dic_windows2[f'w_recon_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])
    dic_windows2[f'w_recon_{subject}'].setHasCursor(0)
    #dic_windows2[f'w_recon_{subject}'].camera(view_quaternion=side_view if side == "L" else middle_view, zoom=0.67)
    #decoded_fname = f"{subject}_{side}_decoded.png"
    #recon_img_path = os.path.join(save_dir, decoded_fname)
    #dic_windows2[f'w_recon_{subject}'].snapshot(recon_img_path, width=1200, height=900)
    #dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])



True


ATransformSet::unregisterObserver: ref 0x5a956dc652a0 not found


True


ATransformSet::unregisterObserver: ref 0x5a956dc652a0 not found


True


ATransformSet::unregisterObserver: ref 0x5a956dc652a0 not found


True


ATransformSet::unregisterObserver: ref 0x5a956dc652a0 not found


no position could be read at 580, 404
no position could be read at 557, 385


In [86]:
for subject in subjects:
    save_dir2 = "/volatile/ad279118/Figures_report/global_reconstruction"
    decoded_fname = f"{subject}_{side}_decoded_view2.png"
    recon_img_path = os.path.join(save_dir2, decoded_fname)
    dic_windows2[f'w_recon_{subject}'].snapshot(recon_img_path, width=1200, height=900)

snap 1 : OpenGL error: invalid operation
snap 1 : OpenGL error: invalid operation


no position could be read at 524, 427
no position could be read at 504, 413
no position could be read at 496, 434
no position could be read at 544, 451
no position could be read at 546, 441
no position could be read at 504, 410
no position could be read at 570, 440
no position could be read at 593, 467
no position could be read at 385, 500
no position could be read at 386, 540
no position could be read at 528, 486
no position could be read at 476, 509
no position could be read at 510, 485
Exiting QApplication


: 

### To save all the individual images

In [33]:
snapshot = True
image_files= []
if snapshot:
    for subject in subjects:
        save_dir = "/volatile/ad279118/Figures_report/global_reconstruction"
        dic_windows2[f'w_recon_{subject}'].setHasCursor(0)
        recon_fname = f"{subject}_{side}_global_reconstruction.png"
        recon_img_path = os.path.join(save_dir, recon_fname)
        dic_windows2[f'w_recon_{subject}'].snapshot(recon_img_path, width=1200, height=900)

        init_fname = f"{subject}_{side}_global_input.png"
        init_img_path = os.path.join(save_dir, init_fname)
        dic_windows2[f'w_init_{subject}'].snapshot(init_img_path, width=1200, height=900)
        image_files.append(init_img_path)
        image_files.append(recon_img_path)

### To save all the individual images within a grid

In [34]:
create_grid(image_files,4,f"{save_dir}/grid_2.png", title='Noillopmahc')

Snapshot of the block available at /volatile/ad279118/Figures_report/global_reconstruction/grid_2.png
